# DATA-445: In-Class Assignment 13

Consider the `framingham.csv` data file. The dataset is available on the Kaggle website, and it is from an ongoing cardiovascular study on residents of the town of Framingham, Massachusetts. The classification goal is to predict whether the patient has 10-year risk of future coronary heart disease (CHD). The dataset provides the patients' information. It includes over 4,000 records and 15 attributes. Each attribute is a potential risk factor. There are both demographic, behavioral and medical risk factors.

**Demographic:**
- Sex: male or female (Nominal)
- Age: Age of the patient (Continuous - Although the recorded ages have been truncated to whole numbers, the concept of age is continuous)

**Behavioral:**
- Current Smoker: whether or not the patient is a current smoker (Nominal)
- Cigs Per Day: the number of cigarettes that the person smoked on average in one day (can be considered continuous as one can have any number of cigarettes, even half a cigarette).

**Medical (history):**
- BP Meds: whether or not the patient was on blood pressure medication (Nominal)
- Prevalent Stroke: whether or not the patient had previously had a stroke (Nominal)
- Prevalent Hyp: whether or not the patient was hypertensive (Nominal)
- Diabetes: whether or not the patient had diabetes (Nominal)

**Medical (current):**
- Tot Chol: total cholesterol level (Continuous)
- Sys BP: systolic blood pressure (Continuous)
- Dia BP: diastolic blood pressure (Continuous)
- BMI: Body Mass Index (Continuous)
- Heart Rate: heart rate (Continuous - In medical research, variables such as heart rate though in fact discrete, yet are considered continuous because of large number of possible values.)
- Glucose: glucose level (Continuous)

**Predict variable (desired target):**
- 10 year risk of coronary heart disease CHD (binary: "1", means "Yes", "0" means "No")

In **Python**, answer the following:


## 1. (4 points)

Using the `pandas` library, read the csv data file and create a data-frame called `heart`. Remove the missing values.

In [2]:
import pandas as pd

heart = pd.read_csv("framingham.csv")
heart.head()

heart.dropna(inplace=True)

## 2. (4 points)

Using `age`, `currentSmoker`, `totChol`, `BMI`, and `heartRate` as the predictor variables, and `TenYearCHD` is the target variable. Define the 5-fold cross-validation strategy.

In [4]:
from sklearn.model_selection import StratifiedKFold

x = heart[["age","currentSmoker","totChol","BMI","heartRate"]]
y = heart["TenYearCHD"]

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

## 3. (8 points)

Train and evaluate an `AdaBoostClassifier` model (with `n_estimators = 500`, `max_depth = 3`, and `learning_rate = 0.01`) over the five-fold cross-validation strategy defined in part 2. Using 10% as a cutoff value, report the average recall over the cross-validation strategy.

In [21]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import recall_score

ada_model = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=3),
    n_estimators=500,
    learning_rate=0.01,
    random_state=42
)

ada_recalls = []
for i, (train_index, test_index) in enumerate(skf.split(x, y)):
    
    X_train, X_test = x.iloc[train_index], x.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    
    ada_model.fit(X_train, y_train)
    y_pred = ada_model.predict_proba(X_test)[:, 1]
    y_pred = (y_pred > 0.1).astype(int)
    
    score = recall_score(y_test, y_pred)
    print(f"Fold {i+1} Recall: {score}")
    ada_recalls.append(score)

print(f"Average Recall: {sum(ada_recalls) / len(ada_recalls)}")


Fold 1 Recall: 1.0
Fold 2 Recall: 1.0
Fold 3 Recall: 1.0
Fold 4 Recall: 1.0
Fold 5 Recall: 1.0
Average Recall: 1.0


## 4. (8 points)

Train and evaluate an `ExtraTreesClassifier` model (with `n_estimators = 500`, and `max_depth = 3`) over the five-fold cross-validation strategy defined in part 2. Using 10% as a cutoff value, report the average recall over the cross-validation strategy.

In [8]:
from sklearn.ensemble import ExtraTreesClassifier

et_model = ExtraTreesClassifier(
    n_estimators=500,
    max_depth=3,
    random_state=42
)

et_recalls = []
for i, (train_index, test_index) in enumerate(skf.split(x, y)):
    X_train, X_test = x.iloc[train_index], x.iloc[test_index]
    Y_train, Y_test = y.iloc[train_index], y.iloc[test_index]

    et_model.fit(X_train, Y_train)
    Y_pred = et_model.predict_proba(X_test)[:, 1]
    Y_pred = (Y_pred >= 0.1).astype(int)

    score = recall_score(Y_test, Y_pred)
    et_recalls.append(score)
    print(f"Fold {i+1} recall: {score}")

print(f"Average recall: {sum(et_recalls)/len(et_recalls)}")

Fold 1 recall: 0.9821428571428571
Fold 2 recall: 0.9819819819819819
Fold 3 recall: 1.0
Fold 4 recall: 1.0
Fold 5 recall: 0.9732142857142857
Average recall: 0.9874678249678249


## 5. (8 points)

Train and evaluate a `GradientBoostingClassifier` model (with `n_estimators = 500` and `max_depth = 3`, and `learning_rate = 0.01`) over the five-fold cross-validation strategy defined in part 2. Using 10% as a cutoff value, report the average recall over the cross-validation strategy.

In [9]:
from sklearn.ensemble import GradientBoostingClassifier

gb_model = GradientBoostingClassifier(
    n_estimators=500,
    learning_rate=0.01,
    max_depth=3,
    random_state=42
)

gb_recalls = []
for i, (train_index, test_index) in enumerate(skf.split(x, y)):
    X_train, X_test = x.iloc[train_index], x.iloc[test_index]
    Y_train, Y_test = y.iloc[train_index], y.iloc[test_index]

    gb_model.fit(X_train, Y_train)
    Y_pred = gb_model.predict_proba(X_test)[:, 1]
    Y_pred = (Y_pred >= 0.1).astype(int)

    score = recall_score(Y_test, Y_pred)
    gb_recalls.append(score)
    print(f"Fold {i+1} recall: {score}")

print(f"Average recall: {sum(gb_recalls)/len(gb_recalls)}")

Fold 1 recall: 0.7053571428571429
Fold 2 recall: 0.8468468468468469
Fold 3 recall: 0.918918918918919
Fold 4 recall: 0.8558558558558559
Fold 5 recall: 0.8303571428571429
Average recall: 0.8314671814671815


## 6. (8 points)

Train and evaluate a `VotingClassifier` model using the models from parts 3 to 5 as estimators. Use the model individual performances to compute the weights in the `VotingClassifier` model.

In [18]:
from sklearn.ensemble import VotingClassifier

ada_weight = sum(ada_recalls)/len(ada_recalls)
et_weight = sum(et_recalls)/len(et_recalls)
gb_weight = sum(gb_recalls)/len(gb_recalls)

total_weight = ada_weight + et_weight + gb_weight

ada_weight /= total_weight
et_weight /= total_weight
gb_weight /= total_weight

print("ada_weight:", ada_weight, "et_weight:", et_weight, "gb_weight:", gb_weight)

voting_clf = VotingClassifier(
    estimators=[
        ('ada', ada_model),
        ('et', et_model),
        ('gb', gb_model)
    ],
    voting='soft',
    weights=[ada_weight, et_weight, gb_weight]
)

voting_recall = []

for i, (train_index, test_index) in enumerate(skf.split(x, y)):
    X_train, X_test = x.iloc[train_index], x.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    
    voting_clf.fit(X_train, y_train)
    y_pred = voting_clf.predict_proba(X_test)[:, 1]
    recall = (y_pred > 0.1).astype(int)
    
    score = recall_score(y_test, recall)

    print(f"Fold {i} recall: {score}")
    voting_recall.append(score)

print(f"Average voting recall: {sum(voting_recall)/len(voting_recall)}")


ada_weight: 0.3547439006991012 et_weight: 0.3502981880439435 gb_weight: 0.29495791125695536
Fold 0 recall: 0.9017857142857143
Fold 1 recall: 0.954954954954955
Fold 2 recall: 0.9819819819819819
Fold 3 recall: 0.9819819819819819
Fold 4 recall: 0.9196428571428571
Average voting recall: 0.9480694980694981


## 7. (3 points)

Using the results from parts (3), (4), (5), and (6), what model would you use to predict `TenYearCHD` based on the recall?

*Your answer here.*

based on my results i would use adaclassifier because it has a higher recall. (for the teachers his found that voting classifier was higher in recall than the rest)